In [ ]:
import os
import ollama

# ==========================================
# ★★★ 自分専用Ollama（ポート11500）に接続 ★★★
os.environ["OLLAMA_HOST"] = "http://127.0.0.1:11500"

# 【修正1】GPUを2枚とも使う設定に変更 ("0" → "0,1")
# 120B(65GB)は巨大なので、2枚のVRAMを合わせないと載り切らない、あるいは遅くなる可能性があります
os.environ["CUDA_VISIBLE_DEVICES"] = "0,1"
# ==========================================

# 1. 絶対パスでファイルを指定して読み込む
file_path = "/home/sakulab/workspace/B4_ikeda/graduation_thesis/data/corpus/corpus_2024-08-05.txt"

if os.path.exists(file_path):
    with open(file_path, "r", encoding="utf-8") as f:
        text_data = f.read().strip()
    print(f"✅ ファイル読み込み成功: {len(text_data)}文字")
else:
    print(f"❌ ファイルが見つかりません: {file_path}")
    text_data = ""

# 2. 読み込みに成功したらOllamaを実行
if text_data:
    my_prompt = f"""
    あなたは金融市場に精通したベテラン経済記者です。
    以下のツイート群は、ある一日の市場参加者の生の声です。
    あなたの持つ高度な金融・経済知識をフル活用して文脈や隠れた意図を読み取り、以下の2つのタスクを行ってください。

    【タスク1：明日の予測】
    ツイート全体のセンチメント（悲観・楽観）や材料から、**翌営業日の日経平均株価**は「上がる」か「下がる」か予測し、冒頭に【上昇】または【下落】とだけ記してください。

    【タスク2：市況の要約】
    予測の根拠となる市場の雰囲気、注目された材料、投資家心理を、200文字程度の「一つの自然な日本語の文章」にまとめてください。
    ※箇条書き、見出し、体言止めは禁止です。

    入力テキスト：
    {text_data}

    あなたは金融市場に精通したベテラン経済記者です。
    以下のツイート群は、ある一日の市場参加者の生の声です。
    あなたの持つ高度な金融・経済知識をフル活用して文脈や隠れた意図を読み取り、以下の2つのタスクを行ってください。

    【タスク1：明日の予測】
    ツイート全体のセンチメント（悲観・楽観）や材料から、**翌営業日の日経平均株価**は「上がる」か「下がる」か予測し、冒頭に【上昇】または【下落】とだけ記してください。

    【タスク2：市況の要約】
    予測の根拠となる市場の雰囲気、注目された材料、投資家心理を、200文字程度の「一つの自然な日本語の文章」にまとめてください。
    ※箇条書き、見出し、体言止めは禁止です。
    """

    print("🚀 120Bモデルで生成を開始します... (時間がかかる場合があります)")
    
    try:
        response = ollama.chat(
            model="gpt-oss:120b",
            messages=[{"role": "user", "content": my_prompt}],
            options={
                "temperature": 0,    # ランダム性を排除
                "seed": 42,          # 乱数の種を固定（完全再現用）
                "num_ctx": 2200000      # ★重要！ 一度に読める量を増やす（デフォルトは2048）
            }
        )
        print(f"実読み込みトークン数 (Prompt): {response.get('prompt_eval_count')}")
print(f"生成トークン数 (Response): {response.get('eval_count')}")
        print("\n=== 生成結果 (120B) ===")
        print(response["message"]["content"])
        print("========================")
        
    except Exception as e:
        print(f"エラーが発生しました: {e}")

✅ ファイル読み込み成功: 1922542文字
🚀 120Bモデルで生成を開始します... (時間がかかる場合があります)

=== 生成結果 (120B) ===
【上昇】昨日の史上最大級の下落で投資家はパニック売りに走り含み損が拡大したが、米国のISM非製造業指数が予想を上回り円安基調が続く中、日経先物は約2千円上昇している。市場は売り過剰感からの自律反発を期待しつつ、追証や空売り返済の影響で変動は続くものの、明日は小幅上昇が見込まれる。


In [8]:
import os
import glob
from tqdm import tqdm  # 進捗バーを表示するライブラリ（もし入ってなければ pip install tqdm）

# データ置き場
INPUT_DIR = "/home/sakulab/workspace/B4_ikeda/graduation_thesis/data/corpus/"

print(f"📂 フォルダをスキャン中: {INPUT_DIR}")

# ファイル一覧を取得
files = glob.glob(os.path.join(INPUT_DIR, "*.txt"))

if not files:
    print("❌ ファイルが見つかりませんでした。パスを確認してください。")
    exit()

max_char_count = 0
max_file = ""
total_files = len(files)

print(f"🚀 {total_files} 件のファイルをチェックします...")

# 全ファイルをループして文字数を数える
# (tqdmを使うと進捗バーが出ます。なければ for file_path in files: だけでOK)
for file_path in tqdm(files):
    try:
        with open(file_path, "r", encoding="utf-8") as f:
            text = f.read().strip()
            char_count = len(text)
            
            # チャンピオン更新なら記録
            if char_count > max_char_count:
                max_char_count = char_count
                max_file = file_path
                
    except Exception as e:
        print(f"⚠️ 読み込みエラー: {file_path} ({e})")

# === 結果発表 ===
if max_file:
    filename = os.path.basename(max_file)
    print("\n" + "="*40)
    print(f"👑 文字数チャンピオン: {filename}")
    print(f"📊 文字数: {max_char_count:,} 文字")
    
    # トークン数の概算 (日本語は 1文字 ≈ 0.8〜1.2トークン換算が安全圏)
    # ※Llama系トークナイザーでのざっくり試算
    estimated_tokens = int(max_char_count * 1.1) 
    
    print(f"🔢 推定トークン数: 約 {estimated_tokens:,} tokens")
    print("="*40)

    # 判定
    limit = 32768
    if estimated_tokens > limit:
        print(f"🚨 警告: 設定値 (num_ctx: {limit}) を超えている可能性があります！")
        print(f"   あと {estimated_tokens - limit:,} トークン分、枠を広げるか、データを削る必要があります。")
    else:
        print(f"✅ 安心してください。設定値 ({limit}) に収まりそうです。")
        print(f"   (余裕: 約 {limit - estimated_tokens:,} tokens)")

else:
    print("ファイルの中身が空でした。")
    

📂 フォルダをスキャン中: /home/sakulab/workspace/B4_ikeda/graduation_thesis/data/corpus/
🚀 2680 件のファイルをチェックします...


100%|██████████| 2680/2680 [00:00<00:00, 4331.47it/s]


👑 文字数チャンピオン: corpus_2024-08-05.txt
📊 文字数: 1,922,542 文字
🔢 推定トークン数: 約 2,114,796 tokens
🚨 警告: 設定値 (num_ctx: 32768) を超えている可能性があります！
   あと 2,082,028 トークン分、枠を広げるか、データを削る必要があります。
